#### EXTRACT `LOCATIONS` FROM VOLUME
- /Volumes/oracle_hrms/landing/operational/locations/

In [0]:
from pyspark.sql.functions import *

locations_df = (
    spark.read.format('csv')
    .option('header', 'true')
    .load('/Volumes/oracle_hrms/landing/operational/locations/*')
)
from pyspark.sql.functions import input_file_name, input_file_block_start, input_file_block_length

trans_locations_df = (
    locations_df
    .selectExpr(
        "LOCATION_ID as location_id",
        "LOCATION_NAME as location_name",
        "LOAD_TS as load_ts",
        "_metadata.file_path as file_path",
        "_metadata.file_name as file_name",
        "_metadata.file_modification_time as load_timestamp"
    )
)


trans_locations_df.display()

In [0]:
trans_locations_df.createOrReplaceTempView('locations_temp_vw')

In [0]:
%skip
%sql
MERGE INTO oracle_hrms.bronze.locations tgt
USING locations_temp_vw src
ON (tgt.location_id = src.location_id 
    AND src.load_timestamp < tgt.load_timestamp)
WHEN MATCHED THEN 
  UPDATE SET
    tgt.location_name = src.location_name,
    tgt.load_ts = src.load_ts,
    tgt.file_path = src.file_path,
    tgt.file_name = src.file_name,
    tgt.load_timestamp = src.load_timestamp
WHEN NOT MATCHED THEN
  INSERT (location_id, location_name, load_ts, file_path, file_name, load_timestamp)
  VALUES (src.location_id, src.location_name, src.load_ts, src.file_path, src.file_name, src.load_timestamp)
;

In [0]:
%sql
-- Get the latest load_timestamp from target table
SELECT MAX(load_timestamp) AS max_load_ts
FROM oracle_hrms.bronze.locations;

-- Perform MERGE to update or insert records
MERGE INTO oracle_hrms.bronze.locations tgt
USING locations_temp_vw src
ON (tgt.location_id = src.location_id)
WHEN MATCHED AND src.load_timestamp > tgt.load_timestamp THEN 
  UPDATE SET
    tgt.location_name   = src.location_name,
    tgt.load_ts         = src.load_ts,
    tgt.file_path       = src.file_path,
    tgt.file_name       = src.file_name,
    tgt.load_timestamp  = src.load_timestamp
WHEN NOT MATCHED THEN
  INSERT (location_id, location_name, load_ts, file_path, file_name, load_timestamp)
  VALUES (src.location_id, src.location_name, src.load_ts, src.file_path, src.file_name, src.load_timestamp);


In [0]:
%sql
-- VALIDATE THE QUERY RESULTS
SELECT
  *
FROM oracle_hrms.bronze.locations
ORDER BY 1;

In [0]:
%sql
DESCRIBE TABLE EXTENDED oracle_hrms.bronze.locations

In [0]:
dbutils.notebook.exit('LOCATIONS LOADED SUCCESSFULLY')